<a href="https://colab.research.google.com/github/lapistach/FeCapSNN/blob/main/Training_FashionMNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount the drive to get access to the datasets and logs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install the dependencies, this is an old version of spiking jelly that was used for the code.

In [ ]:
%%bash
git clone https://github.com/fangwei123456/spikingjelly.git /content/spikingjelly
cd /content/spikingjelly
git reset --hard 73f94ab983d0167623015537f7d4460b064cfca1
pip install .
pip install tensorboard -q

HEAD is now at 73f94ab9 增加detach reset的选项
Processing /content/spikingjelly
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for spikingjelly: filename=spikingjelly-0.0.0.0.1-py3-none-any.whl size=120107 sha256=0c1d6a353d51156641f557c46069b5ca529e4a8084c2483f1b8353c17403ac40
  Stored in directory: /tmp/pip-ephem-wheel-cache-6yfggtdx/wheels/8b/7c/eb/eaababa3ddd15578fcabcc0376ba9473acfed277adc5a871cd
Successfully built spikingjelly
  Attempting uninstall: spikingjelly
    Found existing installation: spikingjelly 0.0.0.0.1
    Uninstalling spikingjelly-0.0.0.0.1:
      Successfully uninstalled spikingjelly-0.0.0.0.1


fatal: destination path '/content/spikingjelly' already exists and is not an empty directory.


The whole cell below is the models.py of the original code by Fang et al. but with only the info needed for Fashion-MNIST.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.clock_driven import functional, layer, surrogate, accelerating
from spikingjelly.clock_driven.neuron import BaseNode, LIFNode
from torchvision import transforms
import math

class PLIFNode(BaseNode):
    def __init__(self, init_tau=2.0, v_threshold=1.0, v_reset=0.0, detach_reset=True, surrogate_function=surrogate.ATan(), monitor_state=False):
        super().__init__(v_threshold, v_reset, surrogate_function, detach_reset, monitor_state)
        init_w = - math.log(init_tau - 1)
        self.w = nn.Parameter(torch.tensor(init_w, dtype=torch.float))

    def forward(self, dv: torch.Tensor):
        if self.v_reset is None:
            self.v += (dv - self.v) * self.w.sigmoid()
        else:
            self.v += (dv - (self.v - self.v_reset)) * self.w.sigmoid()
        return self.spiking()

    def tau(self):
        return 1 / self.w.data.sigmoid().item()

    def extra_repr(self):
        return f'v_threshold={self.v_threshold}, v_reset={self.v_reset}, tau={self.tau()}'

def create_conv_sequential(in_channels, out_channels, number_layer, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
    # 首层是in_channels-out_channels
    # 剩余number_layer - 1层都是out_channels-out_channels
    conv = [
        nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
        nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2)
    ]

    for i in range(number_layer - 1):
        conv.extend([
            nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
            nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2)
        ])
    return nn.Sequential(*conv)


def create_2fc(channels, h, w, dpp, class_num, init_tau, use_plif, alpha_learnable, detach_reset):
    return nn.Sequential(
        nn.Flatten(),
        layer.Dropout(dpp),
        nn.Linear(channels * h * w, channels * h * w // 4, bias=False),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
        layer.Dropout(dpp, dropout_spikes=True),
        nn.Linear(channels * h * w // 4, class_num * 10, bias=False),
        PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
    )


class StaticNetBase(nn.Module):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
        super().__init__()
        self.T = T
        self.init_tau = init_tau
        self.use_plif = use_plif
        self.use_max_pool = use_max_pool
        self.alpha_learnable = alpha_learnable
        self.detach_reset = detach_reset
        self.train_times = 0
        self.max_test_accuracy = 0
        self.epoch = 0
        self.static_conv = None
        self.conv = None
        self.fc = None
        self.boost = nn.AvgPool1d(10, 10)

    def forward(self, x):
        x = self.static_conv(x)
        out_spikes_counter = self.boost(self.fc(self.conv(x)).unsqueeze(1)).squeeze(1)
        for t in range(1, self.T):
            out_spikes_counter += self.boost(self.fc(self.conv(x)).unsqueeze(1)).squeeze(1)

        return out_spikes_counter


class MNISTNet(StaticNetBase):
    def __init__(self, T, init_tau, use_plif, use_max_pool, alpha_learnable, detach_reset):
        super().__init__(T, init_tau=init_tau, use_plif=use_plif, use_max_pool=use_max_pool, alpha_learnable=alpha_learnable, detach_reset=detach_reset)

        self.static_conv = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
        )
        self.conv = nn.Sequential(
            PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(
                tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
            nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2),

            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(
                tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
            nn.MaxPool2d(2, 2) if use_max_pool else nn.AvgPool2d(2, 2)

        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            layer.Dropout(0.5, dropout_spikes=use_max_pool),
            nn.Linear(128 * 7 * 7, 128 * 4 * 4, bias=False),
            PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(
                tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset),
            layer.Dropout(0.5, dropout_spikes=True),
            nn.Linear(128 * 4 * 4, 100, bias=False),
            PLIFNode(init_tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset) if use_plif else LIFNode(
                tau=init_tau, surrogate_function=surrogate.ATan(learnable=alpha_learnable), detach_reset=detach_reset)
        )

class FashionMNISTNet(MNISTNet):
    pass  # 与MNISTNet的结构完全一致

def get_transforms(dataset_name):
    transform_train = None
    transform_test = None
    transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(0.2860, 0.3530),
        ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(0.2860, 0.3530),
    ])

    return transform_train, transform_test

class Interpolate(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.args = args
        self.kwargs = kwargs
    def forward(self, x):
        return F.interpolate(x, *self.args, **self.kwargs)



Below starts the train.py, edited accordingly to the notebook environment and useful only for Fashion-MNIST.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from spikingjelly.clock_driven import functional
import torchvision
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import os
import argparse
import time
import sys

torch.backends.cudnn.benchmark = True
_seed_ = 2020
torch.manual_seed(_seed_)  # use torch.manual_seed() to seed the RNG for all devices (both CPU and CUDA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(_seed_)

In [ ]:
init_tau = 2.0
batch_size = 16
learning_rate = 1e-3
T_max = 64
use_plif = True
alpha_learnable = False
use_max_pool = True
device = 'cuda:0'
dataset_dir = '/content/drive/MyDrive/datasets/FashionMNIST'
log_dir_prefix = '/content/drive/MyDrive/logs'
T = 20
max_epoch = 256
detach_reset = True
number_layer = 5
channels = 128
split_by = 'number'
normalization = None
class_num = 10

In [ ]:
dir_name = f"FashionMNIST_init_tau_{init_tau}_use_plif_{use_plif}_use_max_pool_{use_max_pool}_T_{T}_detach_reset_{detach_reset}"
log_dir = os.path.join(log_dir_prefix, dir_name)
pt_dir = os.path.join(log_dir_prefix, 'pt_' + dir_name)
if not os.path.exists(pt_dir):
        os.mkdir(pt_dir)
print(log_dir, pt_dir)

/content/drive/MyDrive/logs/FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True /content/drive/MyDrive/logs/pt_FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True


In [ ]:
transform_train, transform_test = get_transforms('FashionMNIST')
train_data_loader = torch.utils.data.DataLoader(
    dataset=torchvision.datasets.FashionMNIST(
        root=dataset_dir,
        train=True,
        transform=transform_train,
        download=True),
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last=True,
    pin_memory=True)
test_data_loader = torch.utils.data.DataLoader(
    dataset=torchvision.datasets.FashionMNIST(
        root=dataset_dir,
        train=False,
        transform=transform_test,
        download=True),
    batch_size=batch_size * 16,
    shuffle=False,
    num_workers=4,
    drop_last=False,
    pin_memory=True)

In [ ]:
check_point_path = os.path.join(pt_dir, 'check_point.pt')
check_point_max_path = os.path.join(pt_dir, 'check_point_max.pt')
net_max_path = os.path.join(pt_dir, 'net_max.pt')
optimizer_max_path = os.path.join(pt_dir, 'optimizer_max.pt')
scheduler_max_path = os.path.join(pt_dir, 'scheduler_max.pt')
check_point = None
if os.path.exists(check_point_path):
    check_point = torch.load(check_point_path, map_location=device)
    net = check_point['net']
    print(net.train_times, net.max_test_accuracy)
else:
    net = FashionMNISTNet(T=T, init_tau=init_tau, use_plif=use_plif, use_max_pool=use_max_pool,
                                         alpha_learnable=alpha_learnable, detach_reset=detach_reset).to(device)
print(net)

FashionMNISTNet(
  (boost): AvgPool1d(kernel_size=(10,), stride=(10,), padding=(0,))
  (static_conv): Sequential(
    (0): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv): Sequential(
    (0): PLIFNode(
      v_threshold=1.0, v_reset=0.0, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True, learnable=False)
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): PLIFNode(
      v_threshold=1.0, v_reset=0.0, tau=2.0
      (surrogate_function): ATan(alpha=2.0, spiking=True, learnable=False)
    )
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_d

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max)

In [ ]:
if check_point is not None:
        optimizer.load_state_dict(check_point['optimizer'])
        scheduler.load_state_dict(check_point['scheduler'])
        log_data_list = check_point['log_data_list']
        del check_point
else:
    log_data_list = []

if log_data_list.__len__() > 0 and not os.path.exists(log_dir):
    rewrite_tb = True
else:
    rewrite_tb = False

In [ ]:
from tqdm import tqdm


In [ ]:
writer = SummaryWriter(log_dir)

if rewrite_tb:
    for i in range(log_data_list.__len__()):
        for item in log_data_list[i]:
            writer.add_scalar(item[0], item[1], item[2])

if net.epoch != 0:
    net.epoch += 1
ckpt_time = time.time()
for net.epoch in tqdm(range(net.epoch, max_epoch)):
    start_time = time.time()

    log_data_list.append([])
    print(f'log_dir={log_dir}, max_test_accuracy={net.max_test_accuracy}, train_times={net.train_times}, epoch={net.epoch}')

    net.train()
    for img, label in train_data_loader:
        img = img.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        out_spikes_counter = net(img)
        out_spikes_counter_frequency = out_spikes_counter / net.T
        loss = F.mse_loss(out_spikes_counter_frequency, F.one_hot(label, class_num).float())
        loss.backward()
        optimizer.step()
        functional.reset_net(net)
        if net.train_times % 256 == 0:
            accuracy = (out_spikes_counter_frequency.argmax(dim=1) == label).float().mean().item()
            log_data_list[-1].append(('train_accuracy', accuracy, net.train_times))
            log_data_list[-1].append(('train_loss', loss.item(), net.train_times))
        net.train_times += 1
    scheduler.step()

    net.eval()
    with torch.no_grad():
        test_sum = 0
        correct_sum = 0
        for img, label in test_data_loader:
            img = img.to(device)
            label = label.to(device)
            out_spikes_counter = net(img)
            correct_sum += (out_spikes_counter.argmax(dim=1) == label).float().sum().item()
            test_sum += label.numel()
            functional.reset_net(net)
        test_accuracy = correct_sum / test_sum
        print('test_accuracy', test_accuracy)
        log_data_list[-1].append(('test_accuracy', test_accuracy, net.epoch))
        if use_plif:
            plif_idx = 0
            for m in net.modules():
                if isinstance(m, PLIFNode):
                    log_data_list[-1].append(('w' + str(plif_idx), m.w.item(), net.train_times))
                    plif_idx += 1

        print('Writing....')
        for item in log_data_list[-1]:
            writer.add_scalar(item[0], item[1], item[2])
        if net.max_test_accuracy <= test_accuracy:
            print('save model with test_accuracy = ', test_accuracy)
            net.max_test_accuracy = test_accuracy
            torch.save({
                'net': net,
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'log_data_list': log_data_list
            }, check_point_max_path)

        torch.save({
            'net': net,
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'log_data_list': log_data_list
        }, check_point_path)

        if (time.time() - ckpt_time) / 3600 > 4:
            torch.save({
                'net': net,
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'log_data_list': log_data_list
            }, os.path.join(pt_dir, f'check_point_{net.epoch}.pt'))

        print('Written.')

        speed_per_epoch = time.time() - start_time
        print('speed per epoch', speed_per_epoch)

writer.close()

  0%|          | 0/256 [00:00<?, ?it/s]

log_dir=/content/drive/MyDrive/logs/FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True, max_test_accuracy=0, train_times=0, epoch=0
test_accuracy 0.8914
Writing....
save model with test_accuracy =  0.8914


  0%|          | 1/256 [04:51<20:38:08, 291.33s/it]

Written.
speed per epoch 291.32627606391907
log_dir=/content/drive/MyDrive/logs/FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True, max_test_accuracy=0.8914, train_times=3750, epoch=1
test_accuracy 0.8975
Writing....
save model with test_accuracy =  0.8975


  1%|          | 2/256 [09:42<20:33:05, 291.28s/it]

...

Written.
speed per epoch 289.8123812675476
log_dir=/content/drive/MyDrive/logs/FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True, max_test_accuracy=0.944, train_times=952500, epoch=254
test_accuracy 0.9355
Writing....


100%|█████████▉| 255/256 [20:33:33<04:49, 289.11s/it]

Written.
speed per epoch 289.18886733055115
log_dir=/content/drive/MyDrive/logs/FashionMNIST_init_tau_2.0_use_plif_True_use_max_pool_True_T_20_detach_reset_True, max_test_accuracy=0.944, train_times=956250, epoch=255
test_accuracy 0.937
Writing....


100%|██████████| 256/256 [20:38:29<00:00, 290.27s/it]

Written.
speed per epoch 296.0168581008911
